In [2]:
from bs4 import BeautifulSoup
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import time

#Launches an undetected ChromeDriver instance for automating browser interactions.
driver = uc.Chrome()

#Navigates to the login page
driver.get("https://www.ziprecruiter.in/profiles/sign_in")

#Automates the login process
driver.find_element(By.ID, "profile_email").send_keys("xz134845@gmail.com")
driver.find_element(By.ID, "profile_password").send_keys("xyz12345!")
driver.find_element(By.CSS_SELECTOR, ".btn.btn-lg.btn-success.pull-right").click()

#clicks the navigation item (Suggested jobs) to go to the job listings page.
driver.find_element(By.CLASS_NAME, "nav-item").click()

#`driver.page_source` contains the HTML content
soup = BeautifulSoup(driver.page_source, 'html.parser')

#Finds all the job cards
job_cards = soup.find_all('div', class_='card zip-backfill-link')

#Initialize lists to store job details
job_titles = []
job_companies = []
job_locations = []
job_descriptions = []
job_links = []
job_dates_posted = []

#Loops through each job card and extract details such as job title, company, location, job description, job link, and posted date
for card in job_cards:
    #Extracts job title
    title = card.find('span', class_='card_job_title')
    job_titles.append(title.text.strip() if title else 'N/A')

    #Extracts company name
    company = card.find('span', class_='card_job_company')
    job_companies.append(company.text.strip() if company else 'N/A')

    #Extracts job location
    location = card.find('span', class_='card_job_location')
    job_locations.append(location.text.strip() if location else 'N/A')

    #Extracts job description
    description = card.find('div', class_='card_job_desc')
    job_descriptions.append(description.text.strip() if description else 'N/A')

    #Extracts job link
    link = card.find('a', class_='suggested_job_link')
    job_link = link['href'] if link else 'N/A'
    job_links.append(job_link)


    # Extract the "date posted" by navigating to the job link
    if job_link != 'N/A':
        driver.get(job_link)  #Navigates to the job details page
        details_soup = BeautifulSoup(driver.page_source, 'html.parser')
        date_posted = details_soup.find('div', class_='text-muted')  
        job_dates_posted.append(date_posted.text.strip() if date_posted else 'N/A')

        driver.back()  #Navigates back to the main page
    else:
        job_dates_posted.append('N/A')

#Create a DataFrame to organize the data
jobs_df = pd.DataFrame({
    'Title': job_titles,
    'Company': job_companies,
    'Location': job_locations,
    'Description': job_descriptions,
    'Link': job_links,
    'Date Posted': job_dates_posted 
})

#Display the DataFrame
print(jobs_df)

#Save to an Excel file
jobs_df.to_excel('recent_job_listings.xlsx', engine='openpyxl')

driver.quit()


                                                Title  \
0   Customer Service Associate - Italian Language ...   
1   Senior Officer, Specialist, Customer Service, ...   
2                         Associate, Client Service I   
3                                 Executive - SAP-BTP   
4                                  Project Management   
5                           Lead - Account Management   
6                      Talent Operations Data Analyst   
7                         Business Consulting Analyst   
8                          Data Research Analyst, SRP   
9                    E T Consultant-Data Analyst, GPE   
10                                 Accounts Executive   
11                  Sales Manager-Personal Loan-Sales   
12                                   Graphic Designer   
13                                  Software Engineer   
14                                  Software Engineer   
15                                  Software Engineer   
16                             